In [1]:
import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo = '/content/PhysREVE'
    if not os.path.exists(repo):
        subprocess.check_call(['git', 'clone', '-q', 'https://github.com/UgoBruzadin/PhysREVE.git', repo])
    else:
        subprocess.check_call(['git', '-C', repo, 'pull', '-q'])
    if repo not in sys.path:
        sys.path.insert(0, repo)
    # Colab already ships torch, numpy, scipy, sklearn, matplotlib, seaborn, tqdm, requests
    # Install only what's missing
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mne>=1.6', 'moabb>=1.0', 'xgboost'])
    print('Colab environment ready.')
else:
    print('Local environment — ensure you ran: pip install -e .')

Colab environment ready.


In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from physreve import (
    PhysREVEConfig,
    LabeledEEGDataset, make_split_loaders,
    run_baseline_finetune, run_mae_pretraining, run_pretraining, run_finetuning,
    extract_features, run_ml_baselines,
    compare_models,
)
from physreve.physics import build_leadfield, motor_roi_indices

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [ ]:
# Core
import numpy as np
import torch
import torch.nn as nn

# Analysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# RSA
from scipy.spatial.distance import pdist, squareform

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# ADD: synthetic dataset generator
def generate_sine_wave(freq, fs=256, duration=1.0, noise_std=0.1):
    t = np.linspace(0, duration, int(fs * duration), endpoint=False)
    signal = np.sin(2 * np.pi * freq * t)
    noise = np.random.normal(0, noise_std, size=signal.shape)
    return signal + noise

In [ ]:
def generate_burst(freq, fs=256, duration=1.0, noise_std=0.1):
    t = np.linspace(0, duration, int(fs * duration), endpoint=False)
    signal = np.zeros_like(t)

    mask = (t > 0.3) & (t < 0.6)
    signal[mask] = np.sin(2 * np.pi * freq * t[mask])

    noise = np.random.normal(0, noise_std, size=signal.shape)
    return signal + noise

In [4]:
# ADD: dataset creation
freqs = [4, 8, 10, 20, 40]
X, y = [], []

for f in freqs:
    for _ in range(200):
        X.append(generate_sine_wave(f))
        y.append(f)

X = np.array(X)
y = np.array(y)

# Reshape for EEG-like input: (batch, channels, time)
X = X[:, np.newaxis, :]  # Add channel dimension
print(f'Dataset: {X.shape} (samples, channels, time points)')


# Reshape for EEG-like input: (batch, channels, time)
X = X[:, np.newaxis, :]  # Add channel dimension
print(f'Dataset: {X.shape} (samples, channels, time points)')


In [5]:
# MODIFY: ensure encoder-only forward
with torch.no_grad():
    Z = model.encode(torch.tensor(X).float().to(device))

NameError: name 'model' is not defined